In [4]:
import pandas as pd
from dotenv import load_dotenv
import os 

In [5]:
import sys
sys.path.append("..")

In [6]:
from utils.get_market_data import get_candles

In [22]:
from datetime import datetime, timedelta
now = datetime.now()
from_ = now - timedelta(days=90)
sber_price = get_candles("SBER", from_, now, interval=1)
sber_price.head()

Number of deleted duplicates: 0


,open,close,high,low,value,volume,end
timestamp,,,,,,,
2026-02-20 14:30:00,313.53,313.58,313.59,313.51,2012832.22,6419,2026-02-20 14:30:59
2026-02-20 14:31:00,313.58,313.63,313.63,313.57,5520541.61,17604,2026-02-20 14:31:59
2026-02-20 14:32:00,313.63,313.66,313.7,313.62,4246550.43,13538,2026-02-20 14:32:59
2026-02-20 14:33:00,313.67,313.67,313.7,313.64,2628514.96,8380,2026-02-20 14:33:59
2026-02-20 14:34:00,313.67,313.62,313.69,313.61,2219364.07,7076,2026-02-20 14:34:59


In [8]:
load_dotenv()
db_url = os.getenv("DB_URL")

In [18]:
import psycopg2

conn = psycopg2.connect(db_url)
cur = conn.cursor()

ticker = "SR320CE6"

cur.execute("""
SELECT timestamp, bids, asks FROM orderbooks
WHERE ticker = %s
""", (ticker,))

rows = cur.fetchall()

In [19]:
df = pd.DataFrame(rows, columns=['timestamp', 'bids', 'asks'])
df.head()

,timestamp,bids,asks
0,2026-04-15 07:02:41+00:00,"[{'price': 3.47, 'quantity': 2500}, {'price': ...","[{'price': 11.29, 'quantity': 2500}]"
1,2026-04-15 07:45:33+00:00,"[{'price': 3.5, 'quantity': 30}, {'price': 3.4...","[{'price': 10.28, 'quantity': 2500}, {'price':..."
2,2026-04-15 09:03:00+00:00,"[{'price': 3.5, 'quantity': 30}, {'price': 3.4...","[{'price': 10.28, 'quantity': 2500}, {'price':..."
3,2026-04-15 12:20:40+00:00,"[{'price': 3.5, 'quantity': 30}, {'price': 3.4...","[{'price': 10.28, 'quantity': 2500}, {'price':..."
4,2026-04-15 12:20:41+00:00,"[{'price': 3.5, 'quantity': 30}, {'price': 3.4...","[{'price': 11.9, 'quantity': 800}]"


In [21]:
df['best_bid'] = df['bids'].apply(lambda x: x[0]['price'] if x else None)
df['best_ask'] = df['asks'].apply(lambda x: x[0]['price'] if x else None)

df['mid'] = ((df['best_ask'] + df['best_bid']) / 2).fillna(df['best_ask']).fillna(df['best_bid'])
df = df[['timestamp', 'best_bid', 'best_ask', 'mid']]
df.head(5)

,timestamp,best_bid,best_ask,mid
0,2026-04-15 07:02:41+00:00,3.47,11.29,7.38
1,2026-04-15 07:45:33+00:00,3.50,10.28,6.89
2,2026-04-15 09:03:00+00:00,3.50,10.28,6.89
3,2026-04-15 12:20:40+00:00,3.50,10.28,6.89
4,2026-04-15 12:20:41+00:00,3.50,11.90,7.70


In [25]:
sber_price_df = sber_price[['close']]
sber_price_df.head(5)

,close
timestamp,
2026-02-20 14:30:00,313.58
2026-02-20 14:31:00,313.63
2026-02-20 14:32:00,313.66
2026-02-20 14:33:00,313.67
2026-02-20 14:34:00,313.62
